# 08 - FINAL-TOKEN REPAIR (resumable)

Regenerates the causal endpoints with the preregistered **final-token** A-D
direction (`analysis_plan.md` 4) instead of the mean-of-last-5 `_pooled` one.

**Checkpoints after EVERY branch** - `git commit + push` + a Drive copy. If the
runtime dies, just re-run: `v2_pipeline` skips branches whose bound output already
exists, so a fresh session resumes from git.

**Two-session plan** (each job is idempotent):
1. **GENERATE** - held-out + 5-fold cross-fit, 4 branches (~2 h). Persists per branch.
2. **JUDGE + POST** - StrongREJECT + WildGuard on the pushed generation, then the
   final-token endpoints (~2-3 h). Re-pulls session 1's work.

`RUN_FULL_AD = False` (optional sensitivity table - run later or omit).
No steering, no `--direction-from`, no transfer/MATS.


## 1. Clone + pin + deps (torchao fix)


In [ ]:
PINNED_COMMIT = "4e74434edcbba4b64a5e099eea2adda48ffe38b0"
import os, subprocess, sys, glob, json
from pathlib import Path
REPO = "https://github.com/urosavurdic/dpo-safety-representations.git"
if not os.path.isdir("dpo-safety-representations"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("dpo-safety-representations")
subprocess.run(["git", "fetch", "--all"], check=True)
# work ON the branch (not detached) so per-branch commits push cleanly
BR = "agent/c-quadrant-end-to-end-e0e2317a"
subprocess.run(["git", "checkout", "-B", BR, "origin/" + BR], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BR], check=False)
print("HEAD:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
# Colab ships torchao 0.10 -> peft>=0.19 hard-raises in dispatch_torchao even though
# these LoRA adapters have no torchao layers. Remove it (find_spec None -> gate off).
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
import importlib.util
if importlib.util.find_spec("torchao") is not None:
    import peft.import_utils as _piu
    _piu.is_torchao_available = lambda *a, **k: False
    try:
        import peft.tuners.lora.torchao as _plt
        _plt.is_torchao_available = lambda *a, **k: False
    except Exception:
        pass
import torch, transformers, peft
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      (torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""),
      "| transformers", transformers.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available(), "No GPU - set the Colab runtime to a T4/GPU."


## 2. Drive + HF auth


In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
cands = ["/content/drive/MyDrive/dpo_v2"]
cands += sorted(glob.glob("/content/drive/.shortcut-targets-by-id/*/dpo_v2"))
cands += sorted(glob.glob("/content/drive/Shareddrives/*/dpo_v2"))
REAL = next((c for c in cands if glob.glob(os.path.join(c, "results", "activations", "*_final.npy"))), None)
assert REAL, "no dpo_v2 with activations:\n  " + "\n  ".join(cands)
os.environ["DPO_DRIVE_ROOT"] = REAL
print("Drive:", REAL)
from src.colab_persist import bind, status_line
print(status_line(bind(persist_hf_cache=False)))
try:
    _tok = userdata.get("HF_TOKEN"); os.environ["HF_TOKEN"] = _tok
    from huggingface_hub import login; login(token=_tok); print("HF login OK (job D)")
except Exception as ex:
    print("HF_TOKEN not set - fine for GENERATE; JUDGE needs it:", ex)


## 3. Fetch + SHA-verify inputs; persistence helpers


In [ ]:
import hashlib, shutil, numpy as np
FROZEN_BENCH = "e4946b070f441c7a0676db830c65257b78a2d1b46abb0a61cce4cc86352f838b"
BRANCHES = ["M3", "M3_alt", "M3_direct", "M3_direct_alt"]

def _sha(p):
    h = hashlib.sha256(); h.update(Path(p).read_bytes()); return h.hexdigest()

def pull(rel):
    s, d = Path(REAL) / rel, Path(rel)
    if s.exists() and (not d.exists() or s.stat().st_size != d.stat().st_size):
        d.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(s, d)
    return d.exists()

missing = []
for rel in ["data/frozen_v2/benchmark_v2_20260826T212909Z.jsonl",
            "data/frozen_v2/LATEST_BENCHMARK.json", "logs/direction_split_manifest.json"]:
    pull(rel)
    if not Path(rel).exists(): missing.append(rel)
assert _sha("data/frozen_v2/benchmark_v2_20260826T212909Z.jsonl") == FROZEN_BENCH
for st in sorted(set(BRANCHES) | {"M2", "M3", "M2_alt", "M3_alt"}):
    for kind in ("final", "pooled"):
        rel = "results/activations/" + st + "_" + kind + ".npy"
        pull(rel)
        if not Path(rel).exists(): missing.append(rel); continue
        n = np.load(rel, mmap_mode="r").shape[0]
        if n != 654: missing.append(rel + " (has " + str(n) + " rows, need 654)")
    for rel in ("results/activations/" + st + "_metadata.json",
                "results/activations/" + st + "_metadata_binding.json"):
        pull(rel)
        if not Path(rel).exists(): missing.append(rel)
if missing:
    print("MISSING:\n  " + "\n  ".join(missing))
    raise SystemExit("fix inputs before running any GPU job")
print("inputs OK")

# ---------- checkpoint helpers ----------
# bind() symlinks results/ into <DPO_DRIVE_ROOT>/results, so every generated
# file is ALREADY on the shared Drive folder the moment it is written. The
# only extra durability step is an OPTIONAL git push (needs a GitHub token in
# the clone URL; harmless local commit otherwise).
def persist_git(label):
    subprocess.run(["git", "config", "user.email", "noreply@anthropic.com"], check=True)
    subprocess.run(["git", "config", "user.name", "final-token-repair (Colab)"], check=True)
    add = []
    for pat in ["results/raw/causal_ablation_v2_*_finaltoken*.json",
                "results/refusal_direction/*_final_token*.npy",
                "results/refusal_direction/*_final_token*binding.json",
                "results/final_token_repair/directions/*.npy",
                "results/final_token_repair/bindings/*.json",
                "results/final_token_repair/summaries/*.json",
                "results/final_token_repair/manifests/*.json"]:
        add += [q for q in glob.glob(pat) if Path(q).is_file()]
    add = sorted(set(q for q in add if "judges/" not in q and "behavioral_judges_v2_" not in q))
    if not add:
        print("  persist[" + label + "]: nothing"); return
    subprocess.run(["git", "add", "--"] + add, check=True)
    staged = subprocess.check_output(["git", "diff", "--cached", "--name-only"], text=True).strip()
    if not staged:
        print("  persist[" + label + "]: no change"); return
    msg = "final-token repair (Colab): " + label + chr(10) + chr(10)
    msg += "Co-Authored-By: Claude Sonnet 5 <noreply@anthropic.com>"
    subprocess.run(["git", "commit", "-m", msg], check=True)
    try:
        subprocess.run(["git", "pull", "--rebase", "origin", BR], check=False)
        subprocess.run(["git", "push", "origin", BR], check=True, capture_output=True)
        print("  persist[" + label + "]: pushed " + str(len(staged.splitlines())) + " files to GitHub")
    except Exception:
        print("  persist[" + label + "]: committed locally; git push needs a GitHub token -")
        print("     NOT a problem: the files are on the shared Drive folder via the results/ symlink.")


## 4. CPU: final-token directions + CF3 (torch-free, fast)


In [ ]:
have654 = [b for b in BRANCHES if np.load("results/activations/" + b + "_final.npy", mmap_mode="r").shape[0] == 654]
subprocess.run([sys.executable, "-m", "src.analysis.final_token_repair",
                "--stages", "M2", "M3", "M2_alt", "M3_alt"] + [b for b in have654 if b not in ("M3", "M3_alt")]
               + ["--recompute-cf3", "--out-dir", "results/final_token_repair/summaries"], check=True)
cf3 = json.load(open("results/final_token_repair/summaries/final_token_cf3.json"))
print("CF3 final-token: M2", round(cf3["M2"]["macro_f1"], 4), "M3", round(cf3["M3"]["macro_f1"], 4),
      "cf3", round(cf3["cf3_macroF1_M3_minus_M2"], 4),
      "CI", [round(cf3["bootstrap_group_diff"]["ci_low"], 4), round(cf3["bootstrap_group_diff"]["ci_high"], 4)])
persist_git("CPU final-token directions + CF3")


## SESSION 1 - GENERATE (held-out + 5-fold cross-fit, 4 branches)

Persists after every branch. Re-running skips branches already bound in git.
Watch the per-branch seconds - if the first branch is > ~35 min the T4 is slow;
let it run as far as it gets, it's all checkpointed.


In [ ]:
import time

def gen(stage, extra, label):
    t = time.time()
    subprocess.run([sys.executable, "-m", "src.analysis.v2_pipeline", "causal",
                    "--stage", stage, "--pooling", "final_token"] + extra, check=True)
    print("  " + stage + " " + label + " -> " + str(round(time.time() - t)) + "s")

for br in BRANCHES:
    gen(br, [], "held-out")
    gen(br, ["--cross-fit", "5"], "cross-fit")
    ho = "results/raw/causal_ablation_v2_" + br + "_L24-28_finaltoken.json"
    xf = "results/raw/causal_ablation_v2_" + br + "_L24-28_xfit5_finaltoken.json"
    raw = json.load(open(ho)); assert len(raw) == 3 * 414, (br, "held-out", len(raw))
    assert sorted({r["stage"] for r in raw}) == [br + "_ft_ablated_AD", br + "_ft_ablated_random", br + "_ft_baseline"]
    xb = json.load(open(xf.replace(".json", "_binding.json")))
    assert xb["cross_fit_k"] == 5 and xb["pooling"] == "final_token" and xb["n_test_rows_total"] == 120
    comm = {f["fold"]: sorted(f["test_record_ids"]) for f in
            json.load(open("results/raw/causal_ablation_v2_" + br + "_L24-28_xfit5_binding.json"))["folds"]}
    assert {f["fold"]: sorted(f["test_record_ids"]) for f in xb["folds"]} == comm, br + " partition mismatch"
    # (already on Drive via the results/ symlink) - optional off-Drive git backup:
    persist_git(br + " - held-out + cross-fit final-token OK")
    print("  === " + br + " checkpointed ===")

print("ALL GENERATION DONE + pushed. Run JOB C (optional) then JUDGE, here or in a fresh session.")


## JOB C (OPTIONAL) - full-A/D final-token sensitivity


In [ ]:
RUN_FULL_AD = False
if RUN_FULL_AD:
    for br in BRANCHES:
        gen(br, ["--all-ad-sensitivity"], "full-A/D")
        raw = json.load(open("results/raw/causal_ablation_v2_" + br + "_L24-28_finaltoken_fullAD.json"))
        assert len(raw) == 3 * 300, (br, len(raw))
        persist_git(br + " - full-A/D final-token")
else:
    print("full-A/D SKIPPED (RUN_FULL_AD=False) - document the decision in the report")


## SESSION 2 - JUDGE (StrongREJECT + WildGuard)

Needs `HF_TOKEN` (allenai/wildguard + google/gemma-2b licences). Judges only the
final-token causal rows. Re-running re-scores from scratch (no resume wired in).


In [ ]:
ft_files = [f for f in sorted(glob.glob("results/raw/causal_ablation_v2_*_L24-28_finaltoken*.json"))
            if not f.endswith("_binding.json")]
assert ft_files, "no final-token generation found - run SESSION 1 first (or git pull)"
entries = [{"response_file": f, "binding_file": f.replace(".json", "_binding.json")} for f in ft_files]
assert all("_finaltoken" in e["response_file"] for e in entries)
Path("results/final_token_repair/manifests").mkdir(parents=True, exist_ok=True)
mp = "results/final_token_repair/manifests/consolidated_judge_final_token.json"
json.dump({"kind": "consolidated_response_manifest", "pooling": "final_token",
           "benchmark_sha256": FROZEN_BENCH,
           "split_manifest_sha256": "880381606de7aa2ffbdb8f7c75303cf4937167ed1a2e1b417afeb33761fcf8f1",
           "entries": entries}, open(mp, "w"), indent=2)
print(len(entries), "final-token response files")

OUT = "results/final_token_repair/judges"
subprocess.run([sys.executable, "-m", "src.analysis.behavioral_judges",
                "--response-manifest", mp, "--run-live", "--require-binding",
                "--reject-legacy", "--out-dir", OUT], check=True)
jf = sorted(glob.glob(OUT + "/behavioral_judges_v2_*.json"))[-1]
print("judged:", jf, "  size", round(os.path.getsize(jf)/1e6, 1), "MB")
print("(written to Drive already via the results/ symlink)")


## POST (CPU) - final-token confirmatory endpoints + comparison


In [ ]:
SUM = "results/final_token_repair/summaries"; Path(SUM).mkdir(parents=True, exist_ok=True)
rp = subprocess.run([sys.executable, "-m", "src.analysis.confirmatory_behavioral_endpoints",
                     "--judged", jf, "--condition-infix", "ft_",
                     "--out", SUM + "/final_token_endpoints.json"], capture_output=True, text=True)
print("endpoints exit:", rp.returncode)
print(rp.stdout[-3500:])
if rp.stderr: print("STDERR:", rp.stderr[-2000:])
assert Path(SUM + "/final_token_endpoints.json").exists(), rp.stderr[-4000:]
e = json.load(open(SUM + "/final_token_endpoints.json"))
for st, blk in e.get("CF2_by_stage", {}).items():
    for pop in ("primary", "cross_fitted"):
        b = blk.get(pop) or {}
        print("  " + st.ljust(14) + pop.ljust(14) + " n=" + str(b.get("n_effective_triples")) +
              " cf2=" + str(b.get("cf2")) + " CI=[" + str(b.get("ci_low")) + "," + str(b.get("ci_high")) + "]")
xc = (e.get("CF2_crossfit_branch_contrasts") or {}).get("factorial_2x2", {})
print("  2x2 interaction:", xc.get("corpus_x_history_interaction"))
try:
    pooled = json.load(open("results/summaries/confirmatory_endpoints.json"))
    rows = []
    for st in e.get("CF2_by_stage", {}):
        for pop in ("primary", "cross_fitted", "full_A_sensitivity"):
            pv = (pooled["CF2_by_stage"][st].get(pop) or {}).get("cf2")
            fv = (e["CF2_by_stage"][st].get(pop) or {}).get("cf2")
            if pv is not None and fv is not None:
                rows.append({"stage": st, "population": pop, "pooled_cf2": pv,
                             "final_token_cf2": fv, "abs_diff": abs(pv - fv)})
    json.dump({"rows": rows}, open(SUM + "/pooled_vs_final_token_CF2.json", "w"), indent=2)
    print("wrote pooled_vs_final_token_CF2.json (" + str(len(rows)) + " rows)")
except Exception as ex:
    print("comparison skipped:", ex)
persist_git("JUDGE + POST - final-token endpoints")


## DONE - what is where


In [ ]:
# results/ is a symlink into <DPO_DRIVE_ROOT>/results, so everything below is
# ALREADY on the shared Drive folder. No tar, no extra copy (saves quota).
fin = [q for pat in ["results/raw/causal_ablation_v2_*_finaltoken*.json",
                     "results/final_token_repair/**/*"]
       for q in glob.glob(pat, recursive=True) if Path(q).is_file()]
tot = sum(os.path.getsize(q) for q in fin) / 1e6
for q in sorted(fin):
    print("  " + str(round(os.path.getsize(q) / 1e6, 2)).rjust(7) + " MB  " + q)
print("\n" + str(len(fin)) + " files, " + str(round(tot, 1)) + " MB total")
print("location: " + REAL + "/results/  (shared Drive folder - same as every other artifact)")
print("\nLocal reproduction: git pull, then re-run the POST cell against the judged file.")
